In [ ]:
# ============================================================
# CONFIGURACION DE CREDENCIALES (HIBRIDO: COLAB SECRETS / LOCAL TXT)
# ============================================================
import os

def load_credentials():
    # 1. Intentar cargar desde Google Colab Secrets (si estamos en Colab)
    try:
        from google.colab import userdata
        keys = ['R2_ACCESS_KEY_ID', 'R2_SECRET_ACCESS_KEY', 'R2_ENDPOINT_URL', 'R2_BUCKET_NAME']
        for k in keys:
            try:
                val = userdata.get(k)
                if val: os.environ[k] = val
            except: pass
    except ImportError:
        pass

    # 2. Si no se cargaron (no estamos en Colab o no hay Secrets), buscar archivo local
    if not os.getenv('R2_ACCESS_KEY_ID'):
        cred_path = '../Keys_and_tokens'
        cred_file = os.path.join(cred_path, 'Credenciales.txt')
        
        if os.path.exists(cred_file):
            with open(cred_file, 'r') as f:
                for line in f:
                    if '=' in line and not line.strip().startswith('#'):
                        parts = line.split('=', 1)
                        key = parts[0].strip()
                        value = parts[1].split('#')[0].strip().strip("'").strip('"')
                        os.environ[key] = value
        else:
            # 3. Si no hay nada, pedir manualmente y guardar
            print(" No se detectaron credenciales (Secrets o TXT).")
            os.makedirs(cred_path, exist_ok=True)
            ak = input("  R2_ACCESS_KEY_ID: ")
            sk = input("  R2_SECRET_ACCESS_KEY: ")
            ep = input("  R2_ENDPOINT_URL: ")
            bn = input("  R2_BUCKET_NAME: ")
            with open(cred_file, 'w') as f:
                f.write(f"R2_ACCESS_KEY_ID = '{ak}'\n")
                f.write(f"R2_SECRET_ACCESS_KEY = '{sk}'\n")
                f.write(f"R2_ENDPOINT_URL = '{ep}'\n")
                f.write(f"R2_BUCKET_NAME = '{bn}'\n")
            os.environ['R2_ACCESS_KEY_ID'] = ak
            os.environ['R2_SECRET_ACCESS_KEY'] = sk
            os.environ['R2_ENDPOINT_URL'] = ep
            os.environ['R2_BUCKET_NAME'] = bn

load_credentials()

R2_ACCESS_KEY_ID     = os.getenv('R2_ACCESS_KEY_ID')
R2_SECRET_ACCESS_KEY = os.getenv('R2_SECRET_ACCESS_KEY')
R2_ENDPOINT_URL      = os.getenv('R2_ENDPOINT_URL')
R2_BUCKET_NAME       = os.getenv('R2_BUCKET_NAME')

CARPETA_DESTINO_R2 = 'estaciones_dagma'
print(f'\n Conectado al Bucket: {R2_BUCKET_NAME}')


# Dióxido de Azufre ($SO_2$)
### Proyecto GeoVision-CLIP Cali | Situación 1

Este notebook documenta el comportamiento del $SO_2$, marcador principal de la actividad industrial en el corredor Cali-Yumbo.

## 1. Contexto Geo-Industrial
El Dióxido de Azufre es un **contaminante primario**. En el Valle del Cauca, sus niveles están estrechamente ligados a:
* **Parque Industrial Yumbo-Acopi:** Quema de combustibles fósiles en calderas y procesos manufactureros.
* **Sector Azucarero:** Quemas controladas de biomasa (caña de azúcar) en el norte del municipio.
* **Transporte:** Uso de Diesel de alto contenido de azufre en vehículos de carga pesada.
* **Unidades:** Microgramos por metro cúbico ($\mu g/m^3$).


## 2. Parámetros de Monitoreo
* **Fuente:** Red DAGMA / SISAIRE.
* **Periodo:** 2018 - 2022 (Serie longitudinal de 5 años).
* **Comportamiento Espacial:** A diferencia del Ozono, el $SO_2$ tiende a concentrarse en "plumas" de contaminación que se desplazan según la dirección del viento (Norte a Sur).
* **Importancia Médica:** Es el precursor de la lluvia ácida y causa irritación inmediata de las vías respiratorias superiores.

## 3. Configuración de Estaciones
Las estaciones con mayor sensibilidad a este gas son **La Flora** y **Base Aérea** por su proximidad a la zona industrial de Yumbo.

In [2]:
import pandas as pd
import glob
import os
import zipfile
import plotly.express as px
import plotly.graph_objects as go

# 1. Descomprimir archivos
with zipfile.ZipFile('SO2DAGMA.zip', 'r') as zip_ref:
    zip_ref.extractall('data_so2')

# 2. Buscar TODOS los CSV sin importar en qué subcarpeta estén escondidos
path = 'data_so2'
all_files = glob.glob(path + "/**/*.csv", recursive=True)

print(f"Archivos encontrados para procesar: {len(all_files)}")

# 3. Leer y concatenar arreglando la codificación (tildes)
df_list = []
for f in all_files:
    # Usamos utf-8-sig por si hay caracteres ocultos en los Excels del gobierno
    temp_df = pd.read_csv(f, encoding='utf-8-sig')
    df_list.append(temp_df)

df_so2 = pd.concat(df_list, ignore_index=True)

# 4. Limpieza Estructural
# Renombramos las columnas incluyendo la estación para evitar errores en las gráficas
df_so2 = df_so2.rename(columns={
    'Nombre de la estación': 'Estacion', # Ajuste vital para tus gráficas
    'Fecha inicial': 'Fecha',
    'SO2': 'Concentracion'
})

df_so2['Fecha'] = pd.to_datetime(df_so2['Fecha'])

# Forzamos a que la columna sea numérica
df_so2['Concentracion'] = pd.to_numeric(df_so2['Concentracion'], errors='coerce')

# --- ¡NUEVO FILTRO DE ANOMALÍAS DE SENSOR! ---
# Eliminamos errores negativos (ej. -999) y picos absurdos de calibración (ej. 9999)
# Mantenemos solo el rango físicamente posible en Cali (0 a 200 µg/m³)
df_so2 = df_so2[(df_so2['Concentracion'] >= 0) & (df_so2['Concentracion'] <= 200)]

# Limpiamos filas que no tengan datos ni de estación ni de concentración
df_so2 = df_so2.dropna(subset=['Estacion', 'Concentracion'])

print(f"Dataset de SO2 limpio y filtrado: {df_so2.shape[0]} registros válidos.")

Archivos encontrados para procesar: 5
Dataset de SO2 limpio y filtrado: 94268 registros válidos.


In [3]:
# Graficamos la distribución por estación. Usamos una muestra si el dataset es muy grande
# para que la gráfica interactiva no congele tu navegador.
muestra_so2 = df_so2.sample(n=min(20000, len(df_so2)), random_state=42)

fig1 = px.box(muestra_so2, x="Estacion", y="Concentracion", color="Estacion",
              title="<b>Perfil de Exposición a SO2: Identificación de Zonas Industriales (2018-2022)</b>",
              labels={"Concentracion": "Concentración de SO2 (µg/m³)", "Estacion": "Estación de Monitoreo"},
              template="plotly_dark",
              points="outliers") # Muestra los picos extremos de contaminación

fig1.update_layout(showlegend=False, xaxis_tickangle=-45)
fig1.show()

## 4. Marco Normativo y Eventos Extremos
Para evaluar el impacto real del $SO_2$ en la salud pública de Cali, los datos crudos deben contextualizarse frente a la legislación colombiana.
* **Resolución 2254 de 2017 (MinAmbiente):** Establece el nivel máximo permisible de $SO_2$ en el aire. Para exposiciones de 1 hora, el umbral de alerta es crítico.
* **Identificación de Plumas:** El $SO_2$ viaja en "plumas" (nubes concentradas) impulsadas por los vientos Alisios que entran por el norte del Valle. Un pico repentino en la estación La Flora, seguido de un pico en Univalle horas después, evidencia el transporte de la contaminación industrial de Yumbo hacia la zona residencial del sur.
* **Suavizado de Tendencias:** Debido a la alta volatilidad del viento, utilizaremos una **Media Móvil (Rolling Average)** para aislar la tendencia a largo plazo (2018-2022) y observar si la industria ha reducido o aumentado sus emisiones.

In [4]:
# Agrupamos los datos calculando el promedio por Mes y por Estación
df_tendencia = df_so2.set_index('Fecha').groupby('Estacion').resample('M')['Concentracion'].mean().reset_index()

fig2 = px.line(df_tendencia, x="Fecha", y="Concentracion", color="Estacion",
               title="<b>Evolución Macroeconómica del SO2: Promedios Mensuales Suavizados</b>",
               labels={"Concentracion": "SO2 Promedio Mensual (µg/m³)", "Fecha": "Línea de Tiempo"},
               template="ggplot2",
               line_shape="spline") # Hace que las líneas sean curvas y elegantes

fig2.update_traces(line_width=2.5)
fig2.show()

/tmp/ipykernel_8152/2035709543.py:2: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



In [5]:
!pip install pyarrow fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.8 MB/s eta 0:00:00


In [6]:
df_so2.to_parquet('cali_so2_2018_2022.parquet', engine='pyarrow', index=False)